# Steller Class Prediction (kaggle Competition)

Competition link : https://www.kaggle.com/competitions/playground-series-s6e6/overview

### Step 1 : import train and test dataset

In [1]:
import pandas as pd

try:
    train_df = pd.read_csv(r"Data\train.csv")
    test_df = pd.read_csv(r"Data\test.csv")
    print("dataset load successfully")
except Exception as e:
    print(f"Error: {e}")

dataset load successfully


In [2]:
# overview of train data
train_df.head(3)

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO


In [2]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 577347 entries, 0 to 577346
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 577347 non-null  int64  
 1   alpha              577347 non-null  float64
 2   delta              577347 non-null  float64
 3   u                  577347 non-null  float64
 4   g                  577347 non-null  float64
 5   r                  577347 non-null  float64
 6   i                  577347 non-null  float64
 7   z                  577347 non-null  float64
 8   redshift           577347 non-null  float64
 9   spectral_type      577347 non-null  object 
 10  galaxy_population  577347 non-null  object 
 11  class              577347 non-null  object 
dtypes: float64(8), int64(1), object(3)
memory usage: 52.9+ MB


In [3]:
train_df.shape

(577347, 12)

In [9]:
train_df.describe()

,id,alpha,delta,u,g,r,i,z,redshift
count,577347.00000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000
mean,288673.00000,181.616673,21.834654,22.441926,21.007273,19.962811,19.378911,19.041136,0.723135
std,166665.86727,96.242941,18.933570,2.018135,1.795426,1.648964,1.580059,1.584365,0.810070
min,0.00000,0.011684,-17.966988,-0.139225,13.535483,12.579407,11.962781,11.682803,-0.009970
25%,144336.50000,132.161499,2.474097,20.977090,19.865005,18.820671,18.306820,17.973192,0.181052
50%,288673.00000,188.681465,21.484412,22.570222,21.467820,20.431153,19.631642,19.188598,0.497525
75%,433009.50000,231.829693,36.988310,23.869103,22.292715,21.164096,20.608191,20.162111,0.881390
max,577346.00000,359.999810,79.158322,28.253263,27.620208,25.254499,27.910853,26.826867,7.010780


In [4]:
# null values in each feature
train_df.isnull().sum()

id                   0
alpha                0
delta                0
u                    0
g                    0
r                    0
i                    0
z                    0
redshift             0
spectral_type        0
galaxy_population    0
class                0
dtype: int64

In [5]:
# null values in the test data
test_df.isnull().sum()

id                   0
alpha                0
delta                0
u                    0
g                    0
r                    0
i                    0
z                    0
redshift             0
spectral_type        0
galaxy_population    0
dtype: int64

In [5]:
test_df.shape

(247435, 11)

In [6]:
train_df["class"].value_counts()

class
GALAXY    377480
QSO       117143
STAR       82724
Name: count, dtype: int64

In [7]:
# remove id column from the train dataset
train_df = train_df.drop("id", axis=1)

# verify
print(train_df.columns)

Index(['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type',
       'galaxy_population', 'class'],
      dtype='object')


In [8]:
cat_cols = []
num_cols = []

for cols in train_df.columns:
    if train_df[cols].dtypes == "object":
        cat_cols.append(cols)
    else:
        num_cols.append(cols)

print("numerical cols = ", num_cols)
print("\ncategorical cols = ", cat_cols)
print(f"\ntotal numerical cols = {len(num_cols)}")
print(f"\ntotal categorical cols = {len(cat_cols)}")

numerical cols =  ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']

categorical cols =  ['spectral_type', 'galaxy_population', 'class']

total numerical cols = 8

total categorical cols = 3


In [9]:
cat_cols.remove("class")
print(cat_cols)

['spectral_type', 'galaxy_population']


In [10]:
from sklearn.model_selection import train_test_split
import mlflow.sklearn
from mlflow.models import infer_signature
y = train_df["class"]
X = train_df.iloc[:,:10]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, shuffle=True)
signature = infer_signature(X_train, y_train)

print(f"X_train: {X_train.shape}\ny_train: {y_train.shape}\nX_test: {X_test.shape}\ny_test: {y_test.shape}")

X_train: (461877, 10)
y_train: (461877,)
X_test: (115470, 10)
y_test: (115470,)


In [11]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_new = le.fit_transform(y_train)
y_test_new = le.transform(y_test)
y_train_new = pd.DataFrame(data=y_train_new, columns=["class"])
y_test_new = pd.DataFrame(data=y_test_new, columns=["class"])

print("encoded y train: \n", y_train_new.head(10))
print("\nencoded y test: \n", y_test_new.head(10))

encoded y train: 
    class
0      2
1      1
2      0
3      0
4      0
5      1
6      1
7      0
8      2
9      0

encoded y test: 
    class
0      1
1      2
2      2
3      0
4      0
5      0
6      0
7      0
8      0
9      0


In [12]:
X_train.columns

Index(['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type',
       'galaxy_population'],
      dtype='object')

In [13]:
X_test.columns

Index(['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type',
       'galaxy_population'],
      dtype='object')

In [14]:
# creating pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

# categorical pipeline
cat_preprocessor = Pipeline(steps=[
    # imputation
    ('impute', SimpleImputer(strategy="most_frequent")),
    # encoding
    ('encode', OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    # scaling
    ('scaler', StandardScaler()),
    # PCA
    ('pca', PCA(n_components=0.95))
])

# numerical pipeline
num_preprocessor = Pipeline(steps=[
    # impute
    ('impute', SimpleImputer(strategy="median")),
    # scaling
    ('scaler', StandardScaler()),
    # PCA
    ('pca', PCA(n_components=0.95))
])

# integrate both
preprocssor = ColumnTransformer(transformers=[
    ('num', num_preprocessor, num_cols),
    ('cat', cat_preprocessor, cat_cols)
])

In [15]:
# use several algorithms for classification
# 1. Logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

lr_model = Pipeline(steps=[
    ('prep', preprocssor),
    ('model', LogisticRegression())
])

# decision tree
dt_model = Pipeline(steps=[
    ('prep', preprocssor),
    ('model', DecisionTreeClassifier())
])

# random forest
rf_model = Pipeline(steps=[
    ('prep', preprocssor),
    ('model', RandomForestClassifier())
])

# lightgbm
lgbm_model = Pipeline(steps=[
    ('prep', preprocssor),
    ('model', LGBMClassifier())
])

# XGboost
xgb_model = Pipeline(steps=[
    ('prep', preprocssor),
    ('model', XGBClassifier())
])


models = {
    "logistic_regression":lr_model,
    "decision_tree":dt_model,
    "random_forest":rf_model,
    "lightgbm":lgbm_model,
    "xgboost":xgb_model
}

In [16]:
# function to evaluate the model
from sklearn.metrics import accuracy_score, classification_report

def evaluate_model(model, y_test, y_pred):
    
    accuracy = accuracy_score(y_test, y_pred, weight="balanced")
    print(f"classfication report of {model}: \n", classification_report(y_test, y_pred))
    return accuracy

In [17]:
# create function to save output file 
def submission_file(test_data, best_model, file_name):
    try:
        ids = test_data['id']
        test_data = test_data.drop("id", axis=1)
        y_pred = best_model.predict(test_data)
        test_preds_decoded = le.inverse_transform(y_pred)
        file = pd.DataFrame({
            "id": ids,
            "class": test_preds_decoded
        }).to_csv(f"Resuls/{file_name}.csv", index=False)
        print(file.head())
        print("file saved succesfully")
    except Exception as e:
        print(f"Error: {e}")

In [18]:
def train_models(models_list, X_train, y_train, X_test, y_test):
    model_result = {}
    for name, model in models_list.items():
        print(f"\ntraining {name}...")
        with mlflow.start_run():
            # train model
            model.fit(X_train, y_train)
            # evaluate model
            print(f"\nevaluating {name}...")
            y_pred = model.predict(X_test)
            acc = evaluate_model(model=model, y_test=y_test, y_pred=y_pred)
            print(f"\naccuracy of {name} = {acc}")
            model_result[name] = acc
            mlflow.log_metric("accuracy", acc)
            mlflow.sklearn.log_model(model, "model", signature=signature)
    
    print(model_result)

In [20]:
mlflow.set_experiment(experiment_name="Steller classification competition")
#mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")
train_models(models_list=models, X_train=X_train, y_train=y_train_new, X_test=X_test, y_test=y_test_new)


MlflowException: API request to http://127.0.0.1:5000/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=Steller+classification+competition (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))